In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from sqlalchemy import create_engine

BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
engine = create_engine(f"sqlite:///{BASE / 'cart2insights.db'}")

order_features = pd.read_sql("SELECT * FROM order_features", engine)
item_features = pd.read_sql("SELECT order_id, price, category FROM item_features", engine)

ALPHA = 0.05
print("order_features", order_features.shape, "| item_features", item_features.shape)

order_features (99441, 25) | item_features (112650, 3)


In [2]:
t = order_features[order_features["delivery_status"].isin(["on_time", "delayed"])].dropna(subset=["review_score"])
on_time = t.loc[t["delivery_status"] == "on_time", "review_score"]
delayed = t.loc[t["delivery_status"] == "delayed", "review_score"]

print("GROUP SUMMARY")
print(f"  on_time: n={len(on_time)}, mean={on_time.mean():.3f}, std={on_time.std():.3f}")
print(f"  delayed: n={len(delayed)}, mean={delayed.mean():.3f}, std={delayed.std():.3f}")

# ---- Assumption checks ----
print("\nASSUMPTIONS")
print("  Independence: each order is a different observation (ok).")
print(f"  Sample size: both groups have thousands of rows, so the t-test is robust to non-normality (central limit theorem).")
lev_stat, lev_p = stats.levene(on_time, delayed)
print(f"  Equal variance (Levene): p = {lev_p:.4g} ->", "equal" if lev_p > ALPHA else "NOT equal, so use Welch's t-test")

# ---- Welch's t-test (does not assume equal variance), one-sided: delayed < on_time ----
t_stat, p_val = stats.ttest_ind(delayed, on_time, equal_var=False, alternative="less")

# ---- Effect size (Cohen's d) ----
pooled_sd = np.sqrt((on_time.var() + delayed.var()) / 2)
cohens_d = (on_time.mean() - delayed.mean()) / pooled_sd

# ---- Robustness check: non-parametric test (review scores are ordinal 1-5) ----
u_stat, u_p = stats.mannwhitneyu(delayed, on_time, alternative="less")

print("\nRESULT")
print(f"  Welch t-statistic = {t_stat:.2f}")
print(f"  p-value           = {p_val:.4g}")
print(f"  Cohen's d         = {cohens_d:.2f}  (0.2 small, 0.5 medium, 0.8 large)")
print(f"  Mann-Whitney p    = {u_p:.4g}  (robustness check)")

print("\nDECISION")
if p_val < ALPHA:
    print(f"  p < {ALPHA}: reject H0. Delayed orders have significantly lower review scores.")
else:
    print(f"  p >= {ALPHA}: fail to reject H0. No significant difference.")

GROUP SUMMARY
  on_time: n=87736, mean=4.296, std=1.145
  delayed: n=7563, mean=2.553, std=1.655

ASSUMPTIONS
  Independence: each order is a different observation (ok).
  Sample size: both groups have thousands of rows, so the t-test is robust to non-normality (central limit theorem).
  Equal variance (Levene): p = 0 -> NOT equal, so use Welch's t-test

RESULT
  Welch t-statistic = -89.77
  p-value           = 0
  Cohen's d         = 1.22  (0.2 small, 0.5 medium, 0.8 large)
  Mann-Whitney p    = 0  (robustness check)

DECISION
  p < 0.05: reject H0. Delayed orders have significantly lower review scores.


In [3]:
# One category per order: the category of the most expensive item
main_cat = (item_features.sort_values("price", ascending=False)
                         .drop_duplicates("order_id")[["order_id", "category"]])

valid = order_features[~order_features["order_status"].isin(["canceled", "unavailable"])
                       & order_features["order_value"].notna()][["order_id", "order_value"]]
a = valid.merge(main_cat, on="order_id", how="inner")

counts = a["category"].value_counts()
keep = [c for c in counts[counts >= 100].index if c != "unknown"]
a = a[a["category"].isin(keep)].copy()
print(f"Categories tested: {len(keep)} | orders used: {len(a)}")

groups = [g["order_value"].values for _, g in a.groupby("category")]

# ---- Assumption checks ----
print("\nASSUMPTIONS")
print("  Independence: each order is a different observation (ok).")
print("  Normality: large groups, so ANOVA is robust (central limit theorem). Order value is right-skewed, so we also test the log of the values and run a non-parametric check.")
lev_stat, lev_p = stats.levene(*groups)
print(f"  Equal variance (Levene): p = {lev_p:.4g} ->", "equal" if lev_p > ALPHA else "NOT equal (a limitation, mention it)")

# ---- One-way ANOVA ----
f_stat, p_val = stats.f_oneway(*groups)

# ---- Effect size (eta squared) ----
grand_mean = a["order_value"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = ((a["order_value"] - grand_mean) ** 2).sum()
eta_sq = ss_between / ss_total

# ---- Robustness checks ----
log_groups = [np.log1p(g) for g in groups]
f_log, p_log = stats.f_oneway(*log_groups)
h_stat, p_kw = stats.kruskal(*groups)

print("\nRESULT")
print(f"  F-statistic        = {f_stat:.2f}")
print(f"  p-value            = {p_val:.4g}")
print(f"  Eta squared        = {eta_sq:.3f}  (0.01 small, 0.06 medium, 0.14 large)")
print(f"  ANOVA on log value = F {f_log:.2f}, p {p_log:.4g}")
print(f"  Kruskal-Wallis p   = {p_kw:.4g}")

print("\nDECISION")
if p_val < ALPHA:
    print(f"  p < {ALPHA}: reject H0. Average order value differs significantly across categories.")
else:
    print(f"  p >= {ALPHA}: fail to reject H0.")

cat_means = (a.groupby("category")["order_value"].agg(["count", "mean", "median"]).round(2)
               .sort_values("mean", ascending=False))
print("\nHIGHEST average order value:")
print(cat_means.head(5))
print("\nLOWEST average order value:")
print(cat_means.tail(5))

Categories tested: 51 | orders used: 95945

ASSUMPTIONS
  Independence: each order is a different observation (ok).
  Normality: large groups, so ANOVA is robust (central limit theorem). Order value is right-skewed, so we also test the log of the values and run a non-parametric check.
  Equal variance (Levene): p = 0 -> NOT equal (a limitation, mention it)

RESULT
  F-statistic        = 208.07
  p-value            = 0
  Eta squared        = 0.098  (0.01 small, 0.06 medium, 0.14 large)
  ANOVA on log value = F 266.17, p 0
  Kruskal-Wallis p   = 0

DECISION
  p < 0.05: reject H0. Average order value differs significantly across categories.

HIGHEST average order value:
                            count     mean   median
category                                           
computers                     181  1286.95  1250.81
home_appliances_2             231   524.24   287.27
agro_industry_and_commerce    182   430.65   410.62
musical_instruments           616   334.40   142.82
small_applia

In [4]:
c = order_features.dropna(subset=["main_payment_type"]).copy()
c["status_group"] = np.where(
    c["order_status"] == "delivered", "delivered",
    np.where(c["order_status"].isin(["canceled", "unavailable"]), "canceled_or_unavailable", "in_progress")
)

ct = pd.crosstab(c["main_payment_type"], c["status_group"])
print("OBSERVED COUNTS")
print(ct, "\n")
print("ROW PERCENTAGES (share of each payment type's orders)")
print((ct.div(ct.sum(axis=1), axis=0) * 100).round(2), "\n")

chi2, p_val, dof, expected = stats.chi2_contingency(ct)

# ---- Assumption checks ----
exp_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
share_low = (expected < 5).mean()
print("ASSUMPTIONS")
print("  Independence: each order is counted once (ok).")
print(f"  Smallest expected count: {expected.min():.2f}")
print(f"  Cells with expected count < 5: {share_low:.0%}  (rule: should be under 20%)")
print("  Assumption", "met" if (share_low < 0.2 and expected.min() >= 1) else "NOT fully met (mention it as a limitation)")

# ---- Effect size (Cramer's V) ----
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

print("\nRESULT")
print(f"  Chi-square = {chi2:.2f}")
print(f"  Degrees of freedom = {dof}")
print(f"  p-value    = {p_val:.4g}")
print(f"  Cramer's V = {cramers_v:.3f}  (0.1 small, 0.3 medium, 0.5 large)")

print("\nDECISION")
if p_val < ALPHA:
    print(f"  p < {ALPHA}: reject H0. Payment method and order status are associated.")
else:
    print(f"  p >= {ALPHA}: fail to reject H0. No significant association.")

OBSERVED COUNTS
status_group       canceled_or_unavailable  delivered  in_progress
main_payment_type                                                 
boleto                                 245      19191          348
credit_card                            859      72825         1291
debit_card                              13       1484           30
voucher                                114       2977           60 

ROW PERCENTAGES (share of each payment type's orders)
status_group       canceled_or_unavailable  delivered  in_progress
main_payment_type                                                 
boleto                                1.24      97.00         1.76
credit_card                           1.15      97.13         1.72
debit_card                            0.85      97.18         1.96
voucher                               3.62      94.48         1.90 

ASSUMPTIONS
  Independence: each order is counted once (ok).
  Smallest expected count: 18.90
  Cells with expected count 

## Statistical analysis summary

| Test | H0 | Statistic | p-value | Effect size | Decision |
|---|---|---|---|---|---|
| Welch t-test | Delayed and on-time orders have equal mean review scores | t = -89.77 | < 0.001 | Cohen's d = 1.22 (large) | Reject H0 |
| One-way ANOVA | Mean order value is equal across product categories | F = 208.07 | < 0.001 | Eta squared = 0.098 (medium to large) | Reject H0 |
| Chi-square | Payment method and order status are independent | χ² = 154.48, df = 6 | 8.7e-31 | Cramér's V = 0.028 (negligible) | Reject H0 |

### Business insights (Observation → Interpretation → Impact)

**1. Delivery delay vs satisfaction**
- Observation: delayed orders average 2.55 stars vs 4.30 for on-time orders (n = 7,563 vs 87,736).
- Interpretation: late delivery is strongly associated with lower satisfaction, a large and practically important effect.
- Impact: improving logistics and setting realistic delivery estimates is the most direct way to raise ratings.

**2. Product category vs order value**
- Observation: average order value differs significantly across 51 categories, from R$1,287 (computers) to R$81 (electronics).
- Interpretation: category explains about 10% of the variation in spend.
- Impact: promote high-value categories to raise revenue per order, and bundle or upsell in low-value categories.

**3. Payment method vs order status**
- Observation: the association is statistically significant but negligible (V = 0.028). Voucher orders are canceled or unavailable 3.6% of the time vs about 1.2% for other methods.
- Interpretation: payment method hardly affects outcomes, apart from voucher orders.
- Impact: investigate the voucher checkout and fulfilment flow.

### Assumptions
- Welch's t-test was used because variances were unequal (Levene p < 0.001). Normality is not required with thousands of observations, and a Mann-Whitney test gave the same conclusion.
- ANOVA variances were unequal, which is a limitation. A log-transformed ANOVA and Kruskal-Wallis both confirmed the result.
- For the chi-square test, rare statuses were grouped into 3 outcomes, so every expected count was at least 18.9.
- With about 100,000 orders, tiny differences become significant, so effect sizes are reported next to every p-value.
- These are associations, not proof of cause.